# Article 3 companion · From text to numbers

**Beyond the Stack Notes — working notebook**  
**Model:** [`HuggingFaceTB/SmolLM2-135M`](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), **base** checkpoint (not Instruct)  
**Scope:** tokenization, vocabulary / token IDs, embedding lookup, an intuition for learned parameters, and the limitation of initial token embeddings.

### What we are investigating

1. How do two everyday work requests become token IDs?
2. What does the model's vocabulary size actually count?
3. What does $E\in\mathbb{R}^{V\times d}$ mean, and what does a lookup return?
4. Where do the numerical embedding values come from?
5. Why can two inputs share the same initial token vector while having different meanings?

**Evidence convention:** outputs from the named checkpoint are **measurements when you run these cells**. Numbers in the tiny matrix and BPE exercise are **illustrative**. Explanations of research or architectures are **general claims** supported by linked references, not proven by one model run.

**Deliberate boundaries:** no attention calculation, no transformer forward pass, no next-token generation, no inference-speed benchmarking, and no backpropagation derivation. This notebook is an explanatory companion, not a test of whether the model can actually perform the requested reviews.

**Run locally:** Follow `README.md` and execute cells from top to bottom. First execution downloads the public model (~270 MB in model weights, plus tokenizer/config). No API key or hosted inference is needed. CPU is deliberately used for compatibility, including a 16 GB Apple Silicon MacBook Air. Model files remain in the external cache; only small excerpts of embedding values are displayed.

In [1]:
import sys
from importlib.metadata import version

print("Python:", sys.version.split()[0])
for package in ("torch", "transformers", "tokenizers", "huggingface-hub"):
    print(f"{package}: {version(package)}")
print("Device for this notebook: CPU")

Python: 3.9.6
torch: 2.8.0
transformers: 4.57.6
tokenizers: 0.22.2
huggingface-hub: 0.36.2
Device for this notebook: CPU


## 1 · Why represent text numerically?

A neural network operates on tensors of numbers. A useful **high-level map** is:

```text
User text
    ↓
Tokenizer (segmentation + vocabulary lookup)   ← investigated here
    ↓
Token IDs                                        ← investigated here
    ↓
Learned token embedding matrix E                ← investigated here
    ↓
Initial token vectors                           ← investigated here
    ↓
Transformer layers / context / position         ← next article
    ↓
Output token scores and selection                ← out of scope
```

A token ID is an **identifier**, not a numerical measure of similarity or meaning. E.g., token 42 is not necessarily “closer in meaning” to token 41 than to token 500. That distinction motivates an embedding table.

## 2 · Tokenization: pieces, vocabulary, and IDs

We use the same pair of prompts throughout:

- **A:** `Review the delivery notification component for refactoring.`
- **B:** `Review the customer churn number for Q3.`

They both begin with `Review`, but ask for different kinds of work. We are **not** measuring how well the model performs either task.

### 2.1 Tiny BPE intuition (illustrative, not SmolLM2's actual merges)

Byte Pair Encoding (BPE) learns reusable subword units by repeatedly merging common adjacent units. The real model's tokenizer includes byte-level handling and pre-tokenization rules. Our short exercise starts from **characters** and demonstrates only the **frequency-based merge idea**.

In [2]:
from collections import Counter

# A miniature toy corpus. These strings and learned merges are NOT
# the tokenizer or merge rules used by SmolLM2.
words = ["low", "lower", "lowest", "slow", "slower"]
pieces = [tuple(word) for word in words]

def adjacent_pair_counts(sequences):
    return Counter(
        (left, right)
        for sequence in sequences
        for left, right in zip(sequence, sequence[1:])
    )

def apply_merge(sequence, pair):
    output = []
    i = 0
    while i < len(sequence):
        if i + 1 < len(sequence) and (sequence[i], sequence[i + 1]) == pair:
            output.append(sequence[i] + sequence[i + 1])
            i += 2
        else:
            output.append(sequence[i])
            i += 1
    return tuple(output)

print("Initial pieces:", pieces)
for step in range(3):
    counts = adjacent_pair_counts(pieces)
    best_pair, frequency = counts.most_common(1)[0]
    pieces = [apply_merge(seq, best_pair) for seq in pieces]
    print(f"Merge {step + 1}: {best_pair} (frequency={frequency}) -> {pieces}")

assert len(pieces) == len(words)

Initial pieces: [('l', 'o', 'w'), ('l', 'o', 'w', 'e', 'r'), ('l', 'o', 'w', 'e', 's', 't'), ('s', 'l', 'o', 'w'), ('s', 'l', 'o', 'w', 'e', 'r')]
Merge 1: ('l', 'o') (frequency=5) -> [('lo', 'w'), ('lo', 'w', 'e', 'r'), ('lo', 'w', 'e', 's', 't'), ('s', 'lo', 'w'), ('s', 'lo', 'w', 'e', 'r')]
Merge 2: ('lo', 'w') (frequency=5) -> [('low',), ('low', 'e', 'r'), ('low', 'e', 's', 't'), ('s', 'low'), ('s', 'low', 'e', 'r')]
Merge 3: ('low', 'e') (frequency=3) -> [('low',), ('lowe', 'r'), ('lowe', 's', 't'), ('s', 'low'), ('s', 'lowe', 'r')]


### What the character-level toy leaves out: bytes

The exercise above intentionally starts from visible **characters** because it makes pair counting and progressive merges easy to follow. SmolLM2's GPT-2-style tokenizer instead uses **byte-level BPE**. This is the same core idea at a lower-level starting point.

A byte contains eight binary digits, so one byte has 256 possible values (`0`–`255`). UTF-8 represents a Unicode code point—often what we casually call a character—using a sequence of one to four bytes. For example, `A` uses one byte, `é` uses two, `你` uses three, and `🙂` uses four. That does **not** mean UTF-8 has only 256 characters: sequences of bytes can represent the Unicode code points supported by UTF-8.

In a byte-level BPE tokenizer, text is first pre-tokenized into manageable pieces, then represented through UTF-8 byte units. BPE counts adjacent pairs and learns useful merges. After the first merge, a current unit may be either one byte or an already merged byte sequence. For a four-byte character such as `🙂`, the tokenizer could learn merges that combine all four bytes into one token, combine only some of them, or use byte-level pieces if no helpful merge exists.

This byte-level base provides a fallback for ordinary unfamiliar Unicode text. The tokenizer can encode it through its bytes rather than replacing it with a generic unknown-character token. When tokenizing a new prompt, however, it does **not** learn new merges or expand its vocabulary; it only applies the merge rules learned earlier during tokenizer training.

The character-level toy is therefore a readable demonstration of the BPE process, not a literal reconstruction of SmolLM2's byte-level vocabulary training.

**Interpretation:** repeated merges construct a vocabulary that includes more than individual characters. Actual byte-level BPE tokenization depends on the tokenizer's trained merge rules, its pre-tokenization, and treatment of special tokens. We therefore **never reuse these toy pieces as real SmolLM2 token IDs**.

Sources: [Sennrich, Haddow & Birch (2016)](https://aclanthology.org/P16-1162/) (subword units and rare words); [SmolLM2 model card](https://huggingface.co/HuggingFaceTB/SmolLM2-135M) and [tokenizer configuration](https://huggingface.co/HuggingFaceTB/SmolLM2-135M/blob/main/tokenizer_config.json).

### 2.2 Load the actual tokenizer

The following results will be **measured from the specific SmolLM2 tokenizer**. Unlike an example using `tiktoken`, the IDs here are suitable for looking up rows in **this model's** embedding matrix.

In [3]:
import torch
from transformers import AutoTokenizer, AutoConfig

CHECKPOINT = "HuggingFaceTB/SmolLM2-135M"
# Pin tokenizer, configuration, and model weights to the same revision.
REVISION = "93efa2f097d58c2a74874c7e644dbc9b0cee75a2"

PROMPT_A = "Review the delivery notification component for refactoring."
PROMPT_B = "Review the customer churn number for Q3."

tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT, revision=REVISION, use_fast=True
)
config = AutoConfig.from_pretrained(CHECKPOINT, revision=REVISION)
print("Tokenizer implementation:", type(tokenizer).__name__)
print("Model architecture:", config.model_type)
print("Resolved model config revision:", getattr(config, "_commit_hash", None))

Tokenizer implementation: GPT2TokenizerFast
Model architecture: llama
Resolved model config revision: 93efa2f097d58c2a74874c7e644dbc9b0cee75a2


### 2.3 Inspect real token pieces and IDs

The token spelling shown by a BPE tokenizer is an internal representation. For example, some tokenizers use a visible marker to represent a leading space. We print both the raw spelling (`repr`) and a decoded display for convenience; decoded **individual** tokens are not always meaningful UTF-8 in every language, because a character can span multiple byte-level tokens.

We turn off automatically added special tokens in this experiment so that input-only segmentation is easy to inspect.

In [4]:
def show_tokens(text, label):
    ids = tokenizer.encode(text, add_special_tokens=False)
    pieces = tokenizer.convert_ids_to_tokens(ids)
    print(f"\n{label} ({len(ids)} tokens): {text!r}")
    print(f"{'position':>8} | {'token ID':>8} | {'raw piece':<22} | decoded piece")
    print("-" * 74)
    for index, (piece, token_id) in enumerate(zip(pieces, ids)):
        decoded_piece = tokenizer.decode([token_id], clean_up_tokenization_spaces=False)
        print(f"{index:>8} | {token_id:>8} | {piece!r:<22} | {decoded_piece!r}")
    assert tokenizer.decode(ids, clean_up_tokenization_spaces=False) == text
    return ids

token_ids_input_a = show_tokens(PROMPT_A, "A — engineering review")
token_ids_input_b = show_tokens(PROMPT_B, "B — churn review")
assert len(token_ids_input_a) >= 5, "Choose a longer example if fewer than five tokens are produced."


A — engineering review (10 tokens): 'Review the delivery notification component for refactoring.'
position | token ID | raw piece              | decoded piece
--------------------------------------------------------------------------
       0 |    21465 | 'Review'               | 'Review'
       1 |      260 | 'Ġthe'                 | ' the'
       2 |     7037 | 'Ġdelivery'            | ' delivery'
       3 |    24408 | 'Ġnotification'        | ' notification'
       4 |     5077 | 'Ġcomponent'           | ' component'
       5 |      327 | 'Ġfor'                 | ' for'
       6 |      930 | 'Ġref'                 | ' ref'
       7 |      471 | 'act'                  | 'act'
       8 |     1522 | 'oring'                | 'oring'
       9 |       30 | '.'                    | '.'

B — churn review (9 tokens): 'Review the customer churn number for Q3.'
position | token ID | raw piece              | decoded piece
------------------------------------------------------------------------

### 2.4 What does vocabulary size mean?

The tokenizer's vocabulary contains assignable token IDs; it is **not a dictionary of word meanings**. The `config.vocab_size` describes the model's configured token-ID capacity (and the expected number of embedding rows). Depending on the tokenizer, there can also be added special tokens and a difference between `tokenizer.vocab_size` and `len(tokenizer)`.

Don't assume these metadata values are always identical across models; inspect them.

In [5]:
print("Tokenizer base vocabulary size:", tokenizer.vocab_size)
print("Tokenizer total size (including added tokens):", len(tokenizer))
print("Model configured vocabulary size:", config.vocab_size)
print("Model hidden_size / token embedding dimension:", config.hidden_size)
print("Special token names:", tokenizer.special_tokens_map)
print("Example BOS/EOS IDs:", tokenizer.bos_token_id, tokenizer.eos_token_id)

assert max(token_ids_input_a + token_ids_input_b) < config.vocab_size
assert len(tokenizer) <= config.vocab_size, "Inspect any tokenizer/model vocabulary mismatch."

Tokenizer base vocabulary size: 49152
Tokenizer total size (including added tokens): 49152
Model configured vocabulary size: 49152
Model hidden_size / token embedding dimension: 576
Special token names: {'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'additional_special_tokens': ['<|endoftext|>', '<|im_start|>', '<|im_end|>', '<repo_name>', '<reponame>', '<file_sep>', '<filename>', '<gh_stars>', '<issue_start>', '<issue_comment>', '<issue_closed>', '<jupyter_start>', '<jupyter_text>', '<jupyter_code>', '<jupyter_output>', '<jupyter_script>', '<empty_output>']}
Example BOS/EOS IDs: 0 0


### 2.5 Do text changes affect tokenization?

We should **measure**, not guess, whether case, whitespace, and technical fragments produce different token counts/IDs.

In [6]:
variants = [
    "Review",
    "review",
    " Review",
    "refactoring",
    " refactoring",
    "Q3",
    " Q3",
]
for text in variants:
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    pieces = tokenizer.convert_ids_to_tokens(token_ids)
    print(f"{text!r:<18} -> {len(token_ids)} tokens: {list(zip(pieces, token_ids))}")

'Review'           -> 1 tokens: [('Review', 21465)]
'review'           -> 1 tokens: [('review', 13286)]
' Review'          -> 1 tokens: [('ĠReview', 6769)]
'refactoring'      -> 3 tokens: [('re', 257), ('fact', 2160), ('oring', 1522)]
' refactoring'     -> 3 tokens: [('Ġref', 930), ('act', 471), ('oring', 1522)]
'Q3'               -> 2 tokens: [('Q', 65), ('3', 35)]
' Q3'              -> 2 tokens: [('ĠQ', 1606), ('3', 35)]


### Why does a leading space change the token ID?

The tokenizer operates on the **exact input sequence**, not on dictionary words. `Review` and ` Review` are different sequences: the second includes a leading space. In this GPT-2-style byte-level BPE tokenizer, the raw token spelling for that space-plus-word pattern is `ĠReview`; `Ġ` is an internal display marker for a leading space.

During tokenizer training, frequent byte sequences can become vocabulary entries. This checkpoint has distinct entries for `Review` and for the combined space-plus-`Review` sequence, so they receive distinct, arbitrary IDs. A tokenizer *could* instead produce a separate space token followed by the `Review` token, but the learned combined entry uses one token position. No vocabulary entry is created while processing this prompt; the pretrained tokenizer only applies its already learned merge rules.

Adding a leading space can also change more than the first piece. For example, the segmentations of `refactoring` and ` refactoring` differ because BPE applies its learned merges to the complete sequence it receives.

## 3 · Embeddings: from token IDs to vectors

An integer ID identifies a vocabulary entry, but it does not provide useful continuous coordinates by itself. A common implementation uses a **learned embedding matrix**.

### 3.1 Read the notation first

$$E\in\mathbb{R}^{V\times d}$$

The notation is not intended to be intimidating. We can read it piece by piece:

- $E$: the embedding matrix (a variable name we chose).
- $\in$: “is an element of” / “belongs to”.
- $\mathbb{R}$: real numbers (for example, $0.2$, $-0.4$, $0$).
- $V$: vocabulary size — the **number of rows**.
- $d$: embedding dimension — the **number of columns / values per token vector**.

So $E\in\mathbb{R}^{6\times4}$ simply means **six rows, four real-valued entries per row**. Understanding notation like this also helps us read modern ML papers and technical docs.

**This is a toy matrix, not trained model data:**

In [7]:
# Six token entries in our toy vocabulary; four dimensions per token.
# The numerical values are entirely illustrative.
toy_vocab = {
    0: "Review", 1: "the", 2: "delivery",
    3: "component", 4: "customer", 5: "churn",
}
E_toy = torch.tensor([
    [ 0.2, -0.1,  0.5,  0.3],  # ID 0
    [ 0.7,  0.3, -0.2,  0.1],  # ID 1
    [-0.4,  0.8,  0.1,  0.6],  # ID 2
    [ 0.6, -0.5,  0.9, -0.3],  # ID 3
    [ 0.3,  0.2, -0.6,  0.8],  # ID 4
    [-0.2,  0.4,  0.7,  0.1],  # ID 5
], dtype=torch.float32)

toy_input_ids = torch.tensor([0, 2, 3])  # corresponding to "Review delivery component"
X_toy = E_toy[toy_input_ids]             # Select three rows - embeddings corresponding to our input.

print("E_toy shape (vocabulary × embedding dimensions):", tuple(E_toy.shape))
print("Input tokens:", [toy_vocab[i] for i in toy_input_ids.tolist()])
print("X_toy shape (input tokens × embedding dimensions):", tuple(X_toy.shape))
print("\nRetrieved 3 × 4 matrix:\n", X_toy)

assert tuple(E_toy.shape) == (6, 4)
assert tuple(X_toy.shape) == (3, 4)
assert torch.equal(X_toy[1], E_toy[2])

E_toy shape (vocabulary × embedding dimensions): (6, 4)
Input tokens: ['Review', 'delivery', 'component']
X_toy shape (input tokens × embedding dimensions): (3, 4)

Retrieved 3 × 4 matrix:
 tensor([[ 0.2000, -0.1000,  0.5000,  0.3000],
        [-0.4000,  0.8000,  0.1000,  0.6000],
        [ 0.6000, -0.5000,  0.9000, -0.3000]])


### 3.2 One row for one token ID; one matrix for the input sequence

For a token ID $t_i$ at input position $i$, the lookup is:

$$x_i = E[t_i]$$

- $t_i$ is the **integer ID** at position $i$.
- $E[t_i]$ means **select row $t_i$ of the matrix** (zero-based indexing in code).
- $x_i$ is the resulting **$d$-dimensional token vector**, not the token ID itself.

For $n$ input tokens, collect their row vectors into the matrix $X$:

$$X\in\mathbb{R}^{n\times d}$$

The toy example deliberately has **different sizes**: vocabulary $V=6$, embedding dimension $d=4$, and current sequence length $n=3$. Thus $E$ is $6\times4$ but $X$ is $3\times4$.

### 3.3 Inspect the **real** SmolLM2 matrix

The checkpoint configuration tells us the expected dimensions; `get_input_embeddings()` gives us the actual parameter table. We do **not** run the transformer, compute attention, or generate output here. Initial download requires internet; after caching, the same files can be reused offline.

In [8]:
from transformers import AutoModelForCausalLM

# Default loading on CPU. No MPS/CUDA required for an embedding lookup.
model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT, revision=REVISION, use_safetensors=True
)
model.to("cpu")
model.eval()

embedding_layer = model.get_input_embeddings()
weight = embedding_layer.weight

print("Embedding layer:", type(embedding_layer).__name__)
print("Actual weight shape:", tuple(weight.shape))
print("Parameter dtype:", weight.dtype)
print("Config vocab_size / hidden_size:", config.vocab_size, config.hidden_size)
print("Model config resolved revision:", getattr(model.config, "_commit_hash", None))
print("Embedding matrix memory (MiB):", round(weight.numel() * weight.element_size() / (1024 ** 2), 2))

assert weight.ndim == 2
assert tuple(weight.shape) == (config.vocab_size, config.hidden_size)
assert weight.shape[0] >= len(tokenizer)

Embedding layer: Embedding
Actual weight shape: (49152, 576)
Parameter dtype: torch.float32
Config vocab_size / hidden_size: 49152 576
Model config resolved revision: 93efa2f097d58c2a74874c7e644dbc9b0cee75a2
Embedding matrix memory (MiB): 108.0


### 3.4 Retrieve five **real** vectors

We use the first five **measured** token IDs from prompt A. Each ID selects one row of the learned embedding matrix. We show only four coordinates of each vector for readability; **the remaining coordinates still exist**.

In [9]:
first_five_ids = token_ids_input_a[:5]
first_five_pieces = tokenizer.convert_ids_to_tokens(first_five_ids)

with torch.no_grad():
    # This performs embedding lookup ONLY, no transformer forward pass.
    five_vectors = embedding_layer(torch.tensor(first_five_ids, dtype=torch.long))
    indexed_vectors = weight[first_five_ids]

assert torch.equal(five_vectors, indexed_vectors)
assert tuple(five_vectors.shape) == (5, config.hidden_size)

print("IDs:", first_five_ids)
print("Pieces:", [repr(piece) for piece in first_five_pieces])
print("Retrieved tensor shape:", tuple(five_vectors.shape))
print("\nFirst 4 of", five_vectors.shape[1], "coordinates per token (rounded for display):")
for piece, token_id, vector in zip(first_five_pieces, first_five_ids, five_vectors):
    coords = [round(float(value), 5) for value in vector[:4]]
    print(f"ID {token_id:>6} | {piece!r:<20} | {coords}")

IDs: [21465, 260, 7037, 24408, 5077]
Pieces: ["'Review'", "'Ġthe'", "'Ġdelivery'", "'Ġnotification'", "'Ġcomponent'"]
Retrieved tensor shape: (5, 576)

First 4 of 576 coordinates per token (rounded for display):
ID  21465 | 'Review'             | [0.07812, -0.13672, -0.11816, 0.04858]
ID    260 | 'Ġthe'               | [-0.15137, 0.01807, -0.07422, 0.06543]
ID   7037 | 'Ġdelivery'          | [0.05981, -0.0166, -0.0332, -0.04565]
ID  24408 | 'Ġnotification'      | [0.0957, -0.0238, 0.02454, 0.04346]
ID   5077 | 'Ġcomponent'         | [0.01123, -0.00443, -0.16406, -0.0332]


**Interpretation:** the model uses learned numerical coordinates, but we generally cannot assign a reliable human-readable label (such as “animal” or “size”) to an arbitrary individual column. Inspecting values cannot, by itself, establish what a dimension means. We can say what the **shape** and **lookup** are; we should not invent semantic interpretations.

### 3.5 Where do these values come from?

In pretraining, embedding weights are model parameters and (unless deliberately frozen) are updated through an optimization process guided by the training objective, along with many other parameters. They are **not manually populated dictionary definitions**.

The tiny exercise below shows only **that an embedding weight can be learned/updated**. It uses an artificial regression target solely to make a parameter update visible. It is **not** SmolLM2 training, does **not** model next-token pretraining, and teaches **no backpropagation derivation**.

In [10]:
# Tiny, artificial learning demonstration: not LLM pretraining.
torch.manual_seed(7)  # Use the same random starting table on every run.

# A trainable 6 × 4 table: toy IDs 0–5 each retrieve a four-value vector.
learnable = torch.nn.Embedding(num_embeddings=6, embedding_dim=4)
optimizer = torch.optim.SGD(learnable.parameters(), lr=0.1)

# A one-item input: ID 2 selects the third row because IDs start at zero.
selected_id = torch.tensor([2], dtype=torch.long)
# An invented desired vector, used only to make one update visible.
artificial_target = torch.tensor([[1.0, 0.0, 0.0, 0.0]])

# Save an independent before-update copy of all six rows.
before = learnable.weight.detach().clone()
optimizer.zero_grad()  # Remove gradients that a previous training step could have left behind.
predicted_vector = learnable(selected_id)  # Retrieve the current vector for ID 2.
# Measure the average squared difference from our artificial target.
loss = torch.nn.functional.mse_loss(predicted_vector, artificial_target)
loss.backward()       # Autograd computes gradients; we don't derive them here.

# The optimizer updates trainable weights.
optimizer.step()
after = learnable.weight.detach().clone()  # the updated embedding matrix

print("Artificial loss (before update):", round(float(loss.item()), 5))
print("Selected row before:", before[2].tolist())
print("Selected row after: ", after[2].tolist())
print("Selected row changed:", not torch.equal(before[2], after[2]))
print("Unused row unchanged:", torch.equal(before[1], after[1]))

assert not torch.equal(before[2], after[2])
assert torch.equal(before[1], after[1])

Artificial loss (before update): 1.6899
Selected row before: [-0.8489086031913757, -1.21692955493927, -0.08102158457040787, 1.3614851236343384]
Selected row after:  [-0.7564631700515747, -1.156083106994629, -0.07697050273418427, 1.2934108972549438]
Selected row changed: True
Unused row unchanged: True


**Training vs. ordinary inference:** the example above explicitly runs a parameter update. The earlier lookup cells use a loaded checkpoint and never call an optimizer; simply retrieving embeddings does **not** retrain the checkpoint. `model.eval()` selects evaluation behavior, but does **not** permanently freeze parameters; `torch.no_grad()` avoids gradient tracking for our lookup.

### 3.6 Initial embeddings do not yet use context

Here, **context** means the surrounding tokens in the same input and their order. Compare our two requests:

```text
Review the delivery notification component for refactoring.
Review the customer churn number for Q3.
↑
same opening token
```

Both start with `Review`, so the tokenizer gives that opening piece the same ID. At the initial lookup stage, the embedding table selects one fixed row using only that ID:

$$x_i = E[t_i]$$

The lookup for `Review` does not inspect `delivery`, `notification`, `customer`, or `churn`. Therefore, in a fixed model, the same ID selects the same initial vector in both requests. The vector can contain learned information associated with `Review`, but it is not yet a representation of what `Review` means in this particular request.

Later processing receives the full sequence and can create context-dependent representations. We do not calculate that processing here; this comparison only establishes the boundary between an initial embedding lookup and the later use of surrounding tokens.

In [11]:
print("Prompt A first piece:", repr(tokenizer.convert_ids_to_tokens(token_ids_input_a[0])))
print("Prompt B first piece:", repr(tokenizer.convert_ids_to_tokens(token_ids_input_b[0])))
print("First token IDs:", token_ids_input_a[0], token_ids_input_b[0])

assert token_ids_input_a[0] == token_ids_input_b[0], "If the tokenizer differs, choose a verified shared token."

with torch.no_grad():
    initial_a = embedding_layer(torch.tensor([token_ids_input_a[0]], dtype=torch.long))[0]
    initial_b = embedding_layer(torch.tensor([token_ids_input_b[0]], dtype=torch.long))[0]

print("Initial token embeddings identical:", torch.equal(initial_a, initial_b))
print("But the remaining IDs differ:", token_ids_input_a[1:] != token_ids_input_b[1:])
assert torch.equal(initial_a, initial_b)
assert token_ids_input_a[1:] != token_ids_input_b[1:]

Prompt A first piece: 'Review'
Prompt B first piece: 'Review'
First token IDs: 21465 21465
Initial token embeddings identical: True
But the remaining IDs differ: True


## 4 · Observations, engineering implications, and the next question

### Evidence ledger for Article 3

| Potential article statement | Evidence in this notebook | Qualification |
|---|---|---|
| A tokenizer splits text into discrete units and IDs | Real prompt A/B token table | Specific to this checkpoint/tokenizer |
| BPE can create larger units by merging frequent adjacent pieces | Toy BPE steps; source paper | Toy begins from characters, unlike this byte-level tokenizer |
| IDs are identifiers, not meaningful magnitudes | Vocabulary explanation and ID table | A conceptual property of the scheme, not a measured semantic result |
| Vocabulary size and embedding dimension determine the embedding-table shape | Config and `weight.shape` assertion | Metadata and **actual** SmolLM2 weights |
| Five token IDs retrieve five embedding vectors | `first_five_ids`, `(5, hidden_size)` assertion | Real lookup, truncated numerical display |
| Embedding values are learned parameters | Model weights inspection and toy optimizer step | Toy update shows a mechanism, not the checkpoint's history |
| Same initial token vector can appear in different contexts | Prompt comparison and equal-vector assertion | **Does not** show contextualized representations |
| Token count can depend on formatting | Variant text tokenization cell | Results are tokenizer-dependent |

### What this evidence does *not* establish

- That the model can correctly review code or analyze churn.
- That a single embedding coordinate has a human-readable semantic meaning.
- That similar token IDs imply similar meanings.
- That all tokenizers use the same vocabulary or BPE implementation.
- That initial token embeddings alone encode word sense in context.
- That we have computed attention or contextual representations.

### Engineering implications

- **Prompt budgeting:** count tokens with the model-compatible tokenizer; words and characters are not reliable proxies.
- **Compatibility:** token IDs belong to a specific tokenizer/vocabulary; use it with the matching embedding table.
- **Representation limits:** token embeddings are the **input**, not the full contextual representations constructed in subsequent computation.

**Question for Article 4:** If the same token appears after different preceding text in two requests, how can later computation use that earlier context to produce different representations for the token?

### Source references

- [SmolLM2-135M model card](https://huggingface.co/HuggingFaceTB/SmolLM2-135M)
- [Model configuration](https://huggingface.co/HuggingFaceTB/SmolLM2-135M/blob/main/config.json)
- [Tokenizer configuration](https://huggingface.co/HuggingFaceTB/SmolLM2-135M/blob/main/tokenizer_config.json)
- [Sennrich et al. (2016): Subword Units](https://aclanthology.org/P16-1162/)
- [Hugging Face Transformers docs](https://huggingface.co/docs/transformers/index)

### Reproducibility record

The saved run records Python/package versions, the pinned full checkpoint revision, token IDs, embedding shape and dtype, and passing assertions. Saved outputs describe this run; rerun all cells in a fresh kernel to reproduce the measurements.